# GEETA 4B — notebook-as-module loader (`_nbmodule`)

All shared Python code for GEETA lives in **`tools/*.ipynb`** notebooks (project rule: no `.py` module files). This notebook defines a meta-path importer that makes every `tools/<name>.ipynb` importable as module **`geeta.<name>`**, so imports work exactly like a normal package:

```python
from geeta.verifier import Verifier
from geeta.leak_guard import LeakGuard
from geeta.model_client import ModelClient, MockModel, get_model
from geeta.tutor_engine import TutorSession
from geeta.tutor_data import ProblemStore, TUTOR_SYSTEM
```

## Bootstrap (put in cell 1 of any project notebook or CLI wrapper)

```python
import json, pathlib
ROOT = pathlib.Path.cwd()   # project root — run from there
_nb = json.loads((ROOT / "tools" / "_nbmodule.ipynb").read_text())
_src = "".join("".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code")
exec(compile(_src, str(ROOT / "tools" / "_nbmodule.ipynb"), "exec"), globals())
bootstrap(ROOT)
```

`install(root)` / `bootstrap(root)` are idempotent and safe to call from multiple notebooks or kernels. Relative imports inside the tool notebooks (e.g. `from .tutor_data import ...`) keep working because `geeta` is registered as a package whose `__path__` points at `tools/`.


In [ ]:
from __future__ import annotations

import importlib
import importlib.abc
import importlib.machinery
import importlib.util
import json
import pathlib
import sys
import types

_PACKAGE = "geeta"
_TOOLS_DIRNAME = "tools"
_MARKER = "_geeta_nb_finding"


class _NBLoader(importlib.abc.Loader):
    """Exec the code cells of a notebook into a module namespace."""

    def __init__(self, nb_path: pathlib.Path):
        self.nb_path = pathlib.Path(nb_path)

    def create_module(self, spec):
        return None  # use default module creation

    def exec_module(self, module):
        nb = json.loads(self.nb_path.read_text(encoding="utf-8"))
        chunks = [
            "".join(cell.get("source", []))
            for cell in nb.get("cells", [])
            if cell.get("cell_type") == "code"
        ]
        source = "\n\n".join(chunks)
        # Cells are joined into ONE file, so any `from __future__ import ...`
        # lines (legal per-cell in Jupyter, illegal mid-file in a module)
        # are hoisted to the top, preserving the original .py semantics.
        import re as _re
        future = []
        kept = []
        for line in source.splitlines(keepends=True):
            if _re.match(r"^\s*from\s+__future__\s+import\s+\S.*$", line):
                names = line.strip()[len("from __future__ import"):].strip()
                if names not in future:
                    future.append(names)
                continue
            kept.append(line)
        if future:
            source = "".join("from __future__ import " + f + "\n" for f in future) + "".join(kept)
        code = compile(source, str(self.nb_path), "exec")
        module.__nb_path__ = str(self.nb_path)  # provenance for debugging
        exec(code, module.__dict__)


class NBPackageFinder:
    """Meta-path finder: geeta.<mod> -> <root>/tools/<mod>.ipynb.

    Installed at the front of ``sys.meta_path`` so notebook modules always win
    over any stray .py files of the same name.
    """

    def __init__(self, root: pathlib.Path):
        self.root = pathlib.Path(root)
        self._tools = self.root / _TOOLS_DIRNAME
        setattr(self, _MARKER, True)

    def find_spec(self, fullname, path=None, target=None):
        if fullname == _PACKAGE:
            return None  # package is pre-registered in sys.modules by install()
        if not fullname.startswith(_PACKAGE + "."):
            return None
        sub = fullname.split(".", 1)[1]
        if "/" in sub or "." in sub:
            return None  # only direct tools/<name>.ipynb modules
        nb = self._tools / (sub + ".ipynb")
        if not nb.exists():
            return None
        spec = importlib.machinery.ModuleSpec(fullname, _NBLoader(nb))
        spec.has_location = True
        return spec


def install(root: "str | pathlib.Path" = ".") -> types.ModuleType:
    """Register the notebook-package finder for ``geeta.<mod>`` imports.

    Idempotent: re-calling re-points at ``root`` and deduplicates finders.
    Returns the ``geeta`` package module (a bare namespace with ``__path__``).
    """
    root = pathlib.Path(root).expanduser().resolve()
    tools = root / _TOOLS_DIRNAME
    if not (tools / "_nbmodule.ipynb").exists():
        raise FileNotFoundError(
            f"tools/_nbmodule.ipynb not found under {root} — call install(<project root>)"
        )
    sys.meta_path[:] = [f for f in sys.meta_path if not getattr(f, _MARKER, False)]
    sys.meta_path.insert(0, NBPackageFinder(root))
    pkg = sys.modules.get(_PACKAGE)
    if pkg is None:
        pkg = types.ModuleType(_PACKAGE)
        sys.modules[_PACKAGE] = pkg
    pkg.__doc__ = (
        "GEETA 4B runtime package. Modules are tools/*.ipynb notebooks, "
        "imported as geeta.<name> by tools/_nbmodule.ipynb."
    )
    pkg.__path__ = [str(tools)]
    return pkg


def bootstrap(root: "str | pathlib.Path" = ".") -> types.ModuleType:
    """install() alias — the name project notebooks use in their bootstrap cell."""
    return install(root)


def notebook_modules(root: "str | pathlib.Path" = ".") -> list:
    """List importable geeta.<mod> modules available under root/tools."""
    tools = pathlib.Path(root).expanduser().resolve() / _TOOLS_DIRNAME
    return sorted(p.stem for p in tools.glob("*.ipynb"))
